# Actividad 2

# Información General del Dataset

**Nombre:** *Student Performance Factors Dataset*

**Fuente y liga de acceso:** *https://www.kaggle.com/datasets/mosapabdelghany/student-performance-factors-dataset*

**Fecha de consulta:** *13 de agosto de 2026 — 17 de agosto de 2026*

**Fecha de última actualización en Kaggle:** *Hace 10 meses*

**Descripción:** *Este conjunto contiene variables numéricas y categóricas relacionadas con factores académicos, personales, familiares y esoclares que pueden influir en el desempeño de los estudiantes. En esta actividad se tendrá como objetivo seleccionar las variables más importantes y entrenar los modelos para que puedan ser capaces de predecir cuáles variables son las que verdaderamente influyen.*

# Datos del Dataset

## Explicación de los registros

En este caso se usará el dataset limpio que se obtuvo de la actividad 1. Éstos son los registros originales y nuevos.

**Registros originales**
* **Hours_Studied:** *Número de horas de estudio por semana*
* **Attendance:** *Porcentaje de asistencia en el periodo*
* **Parental_Involvment:** *Nivel de apoyo académico por parte de los padres*
* **Access_to_Resources:** *Disponibilidad de recursos académicos como libros o internet*
* **Extracurricular_Activities:** *Si el estudiante está involucrado en actividades extracurriculares*
* **Sleep_Hours:** *Promedio de horas de sueño al día*
* **Previous_Scores:** *Promedio de calificaciones académicas previas*
* **Motivation_Level:** *Nivel de motivación del estudiante*
* **Internet_Access:** *Si el estudiante tiene acceso al internet*
* **Tutoring_Sessions:** *Número de sesiones de tutoría a la semana*
* **Family_Income:** *Nivel de ingresos familiares*
* **Teacher_Quality:** *Calidad general percibida de los docentes*
* **School_Type:** *Tipo de escuela (Pública o Privada)*
* **Peer_Influence:** *Influencia de los compañeros en el rendimiento académico*
* **Physical_Activity:** *Horas de actividad física a la semana*
* **Learning_Disabilities:** *Si el estudiante presenta dificultades de aprendizaje*
* **Parental_Education_Level:** *Nivel máximo de estudios de los padres*
* **Distance_from_Home:** *Distancia entre el hogar y la escuela (Cerca, Moderada, Lejos)*
* **Gender:** *Género del estudiante*
* **Exam_Score:** *Calificación final del examen*

### Variable objetivo
La variable **Exam_Score** es nuestra variable objetivo.

### Pregunta predictiva que guiará el análisis
¿Qué variables son las fundamentales para que un estudiante pueda obtener un puntaje alto en su examen? ¿Existe una combinación de variables que aumente o disminuya este puntaje?

# Paso 1 y 2

In [ ]:
%pip install matplotlib seaborn pandas scikit-learn kagglehub

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import kagglehub
from kagglehub import KaggleDatasetAdapter

### Limpieza que se realizó en la Actividad 1

In [ ]:
def clean_data():
    # Load the dataset from Kaggle
    file_path = "StudentPerformanceFactors.csv"
    
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "mosapabdelghany/student-performance-factors-dataset",
        file_path,
    )
    
    # Columns of type string with missing values
    df_objects = df.select_dtypes(include=['object'])
    missing_columns = df_objects.columns[df_objects.isnull().any()].tolist()
    
    # Columns of type int64 with missing values
    #df_int64 = df.select_dtypes(include=['int64'])
    #missing_columns += df_int64.columns[df_int64.isnull().any()].tolist()
    
    # Print the number of missing values for each column with missing values
    for column in missing_columns:
        print(f"Valores faltantes en '{column}': {df[column].isnull().sum()}")
    
    # Fill missing values with the mode of the respective columns
    for column in missing_columns:
        df[column] = df[column].fillna(df[column].mode()[0])
        
    # Convert hierarchical categorical variables to numerical using one-hot encoding
    map_level = {
      'Low': 0,
      'Medium': 1,
      'High': 2
    }
    
    hc_columns = ['Parental_Involvement', 'Motivation_Level']
    
    for column in hc_columns:
        df[f'{column}_encoded'] = df[column].map(map_level)
        
    # Convert categorical variables to numerical using one-hot encoding
    df = pd.get_dummies(df, columns=['School_Type', 'Internet_Access'], drop_first=True, dtype=int)
    
    # Clean atypical values in Exam_Score column
    over_100 = (df['Exam_Score'] > 100).sum()
    print(f"\nRegistros con calificación mayor a 100 en 'Exam_Score': {over_100}")
    
    under_100 = (df['Exam_Score'] < 0).sum()
    print(f"Registros con calificación menor a 0 en 'Exam_Score': {under_100}")
    
    df.loc[df['Exam_Score'] > 100, 'Exam_Score'] = 100
    df.loc[df['Exam_Score'] < 0, 'Exam_Score'] = 0
    
    print("\nValor mínimo y máximo de 'Exam_Score' después de la limpieza:")
    print(f"Valor mínimo: {df['Exam_Score'].min()}")
    print(f"Valor máximo: {df['Exam_Score'].max()}")
    
    # Standard scaling of 'Hours_Studied' and 'Attendance'
    cols_to_scale = ['Hours_Studied', 'Attendance']
    scaler = StandardScaler()
    
    for column in cols_to_scale:
        df[f'{column}_scaled'] = scaler.fit_transform(df[[column]])
        
    return df

df = clean_data()

print("Descripción estadística del DataFrame:")
print(f"\nNúmero total de registros: {len(df)}")
print(f"\nNúmero de columnas: {len(df.columns)}")

print("\nValidación de cero valores nulos en el DataFrame:")
print(f"Valores nulos en el DataFrame: {df.isnull().sum().sum()}")

print("\nTipos de datos de las columnas después de la limpieza:")
print(df.dtypes)

print("\nNúmero de registros duplicados en el DataFrame después de la limpieza:")
print(df.duplicated().sum())

df.describe(include="all")

# Conclusión y Justificación de Cambios

**Registros nuevos**
* **Parental_Involvement_Encoded:** *Nivel de apoyo académico por parte de los padres medido en valores númericos (del 0 al 2)*
* **Motivation_Level_Encoded:** *Niveo de motivación del estudiante medido en valores númericos (del 0 al 2)*
* **School_Type_Private:** *Si el estudiante atiende escuela privada (0 siendo Falso y 1 siendo Verdadero)*
* **School_Type_Public:** *Si el estudiante atiende escuela pública (0 siendo Falso y 1 siendo Verdadero)*
* **Internet_Access_No:** *Si el estudiante no posee acceso a internet (0 siendo Falso y 1 siendo Verdadero)*
* **Internet_Access_Yes:** *Si el estudiante sí posee acceso a internet (0 siendo Falso y 1 siendo Verdadero)*
* **Hours_Studied_Scaled:** *Número de horas de estudio por semana estandarizadas. Centrado en 0 con una desviación estándar de 1.*
* **Attendance_Scaled:** *Porcentaje de asistencia en el periodo estandarizado. Centrado en 0 con una desviación estándar de 1.*

(JAIR: AQUÍ ME FALTA PONER LA JUSTIFICACIÓN)

# Paso 3

Explorando la variable 'Exam_Score'

In [ ]:
print("\nEstadística descriptiva de la columna 'Exam_Score':")
df['Exam_Score'].describe()

# Histogram of 'Exam_Score'
plt.figure(figsize=(10, 6))
sns.histplot(df['Exam_Score'], bins=20, kde=True, color='blue')
plt.title('Distribución de Calificaciones de Examen')
plt.xlabel('Calificación de Examen')
plt.ylabel('Frecuencia')
plt.show()

# Boxplot of 'Exam_Score'
plt.figure(figsize=(10, 6))
sns.boxplot(x=df['Exam_Score'], color='lightgreen')
plt.title('Boxplot de Calificaciones de Examen')
plt.xlabel('Calificación de Examen')
plt.show()

In [ ]:
# Correlation heatmap of the numerical columns
plt.figure(figsize=(12, 8))
numerical_columns = df.select_dtypes(include=['number']).columns
correlation_matrix = df[numerical_columns].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Matriz de Correlación')
plt.show()

# EN RESUMEN

Sólo las variables Attendance y Hours_Studied merecen la pena ser exploradas como posibles causas de un mejor resultado en el examen.